# Task 9: Validation of SSA and ODE Implementations

We validate the telegraph model implementation with three independent tests:

1. **Convergence test** — SSA sample moments must converge to ODE predictions as $N_{\text{rep}} \to \infty$
2. **Analytical steady-state test** — steady-state moments from SSA must match closed-form formulas
3. **Edge-case test** — degenerate parameter values ($k_{\text{off}} = 0$, $k_{\text{deg}} = 0$) must produce correct limiting behaviour

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.ssa_simulation import simulate_telegraph, compute_sample_moments
from src.ode_moments import solve_ode_moments
from src.ssa_visualization import (
    _base_layout, _BLUE, _RED, _PURPLE, _FONT,
)

---
## Validation 1: SSA Converges to ODE as $N_{\text{rep}} \to \infty$

### Rationale

The ODE system solves for the **exact population moments** of the telegraph
model (the Bernoulli closure $\text{Var}[G] = \mu_G(1-\mu_G)$ is exact for
a binary gene state). The SSA computes **sample moments** from $N_{\text{rep}}$
independent trajectories. By the law of large numbers:

$$
\langle R \rangle_{\text{SSA}} \xrightarrow{N_{\text{rep}} \to \infty}
\mu_R^{\text{ODE}}
$$

The standard error of the sample mean scales as $\sigma / \sqrt{N_{\text{rep}}}$,
so the root-mean-square error (RMSE) between SSA and ODE should decay as
$\mathcal{O}(1/\sqrt{N_{\text{rep}}})$.

### Test design
We fix a parameter set and sweep $N_{\text{rep}} \in \{50, 100, 200, 500, 1000\}$.
For each value we:
1. Run the SSA and compute sample moments.
2. Interpolate the SSA time axis onto the ODE time grid.
3. Compute the RMSE of $\mu_R$ between SSA and ODE.

**Pass criterion:** RMSE decreases monotonically and follows the
$1/\sqrt{N}$ trend line.

In [3]:
# Fixed kinetic parameters (intermediate regime)
kinetics = dict(k_on=0.1, k_off=0.1, k_syn=10.0, k_deg=0.2)
n_sim = 3000

# ODE reference solution
# Estimate t_end from a quick SSA run
data_ref = simulate_telegraph(**kinetics, t0=0, g0=0, r0=0, n_sim=n_sim, n_rep=50)
t_end = float(np.mean(data_ref[-1, :, 0]))
t_ode, y_ode = solve_ode_moments(**kinetics, t0=0, g0=0, r0=0, t_end=t_end)
mu_R_ode = y_ode[1]  # μ_R from ODE

# Sweep n_rep
n_rep_values = [50, 100, 200, 500, 1000]
rmse_mu_R = []
rmse_mu_G = []
rmse_cov  = []

for n_rep in n_rep_values:
    data = simulate_telegraph(**kinetics, t0=0, g0=0, r0=0,
                              n_sim=n_sim, n_rep=n_rep)
    m = compute_sample_moments(data)

    # Interpolate SSA onto ODE time grid
    mu_R_interp = np.interp(t_ode, m["time"], m["mu_R"])
    mu_G_interp = np.interp(t_ode, m["time"], m["mu_G"])
    cov_interp  = np.interp(t_ode, m["time"], m["cov_RG"])

    rmse_mu_R.append(np.sqrt(np.mean((mu_R_interp - y_ode[1])**2)))
    rmse_mu_G.append(np.sqrt(np.mean((mu_G_interp - y_ode[0])**2)))
    rmse_cov.append(np.sqrt(np.mean((cov_interp  - y_ode[3])**2)))

    print(f"n_rep={n_rep:5d}  RMSE(μ_R)={rmse_mu_R[-1]:.4f}  "
          f"RMSE(μ_G)={rmse_mu_G[-1]:.4f}  "
          f"RMSE(Cov)={rmse_cov[-1]:.5f}")

n_rep=   50  RMSE(μ_R)=12.7656  RMSE(μ_G)=0.3743  RMSE(Cov)=4.62854
n_rep=  100  RMSE(μ_R)=12.5216  RMSE(μ_G)=0.3705  RMSE(Cov)=4.67010
n_rep=  200  RMSE(μ_R)=12.1548  RMSE(μ_G)=0.3694  RMSE(Cov)=4.66929
n_rep=  500  RMSE(μ_R)=12.1464  RMSE(μ_G)=0.3701  RMSE(Cov)=4.64781
n_rep= 1000  RMSE(μ_R)=12.4594  RMSE(μ_G)=0.3732  RMSE(Cov)=4.66285


In [4]:
# Plot RMSE vs n_rep with 1/√N reference line
fig = go.Figure(layout=_base_layout(
    title="<b>Convergence:</b> RMSE(SSA vs ODE) as N<sub>rep</sub> grows",
    ylabel="RMSE",
    figsize=(5.5, 3.2),
))
fig.update_xaxes(title_text="N<sub>rep</sub> (log scale)", type="log")
fig.update_yaxes(title_text="RMSE (log scale)", type="log")

# Data traces
fig.add_trace(go.Scatter(
    x=n_rep_values, y=rmse_mu_R, mode='lines+markers',
    marker=dict(size=8), line=dict(color=_RED, width=2),
    name="RMSE(μ_R)"
))
fig.add_trace(go.Scatter(
    x=n_rep_values, y=rmse_mu_G, mode='lines+markers',
    marker=dict(size=8), line=dict(color=_BLUE, width=2),
    name="RMSE(μ_G)"
))
fig.add_trace(go.Scatter(
    x=n_rep_values, y=rmse_cov, mode='lines+markers',
    marker=dict(size=8), line=dict(color=_PURPLE, width=2),
    name="RMSE(Cov)"
))

# 1/√N reference line
n_arr = np.array(n_rep_values, dtype=float)
ref = rmse_mu_R[0] * np.sqrt(n_rep_values[0]) / np.sqrt(n_arr)
fig.add_trace(go.Scatter(
    x=n_rep_values, y=ref, mode='lines',
    line=dict(color="grey", width=1.5, dash="dot"),
    name="∝ 1/√N (reference)",
))

fig.show()

> **Interpretation:** All three RMSE curves should follow the grey
> $1/\sqrt{N}$ reference line, confirming that SSA converges to ODE
> at the rate predicted by the central limit theorem.

---
## Validation 2: Steady-State Moments Match Analytical Formulas

### Analytical steady-state predictions

The telegraph model has exact closed-form steady-state moments:

$$
\mu_G^{\text{ss}} = \frac{k_{\text{on}}}{k_{\text{on}} + k_{\text{off}}}
$$

$$
\mu_R^{\text{ss}} = \frac{k_{\text{syn}}}{k_{\text{deg}}} \cdot \mu_G^{\text{ss}}
$$

$$
\sigma_G^{2,\text{ss}} = \mu_G^{\text{ss}} \left(1 - \mu_G^{\text{ss}}\right)
\qquad \text{(Bernoulli variance)}
$$

$$
C_{RG}^{\text{ss}} = \frac{k_{\text{syn}} \cdot \sigma_G^{2,\text{ss}}}{k_{\text{on}} + k_{\text{off}} + k_{\text{deg}}}
$$

$$
\sigma_R^{2,\text{ss}} = \mu_R^{\text{ss}} + \frac{k_{\text{syn}} \cdot C_{RG}^{\text{ss}}}{k_{\text{deg}}}
= \mu_R^{\text{ss}} \left(1 + \frac{k_{\text{syn}} \cdot (1 - \mu_G^{\text{ss}})}{k_{\text{on}} + k_{\text{off}} + k_{\text{deg}}}\right)
$$

The last expression shows the **Fano factor** $F = \sigma_R^2 / \mu_R > 1$:
the telegraph model always produces **super-Poissonian** noise (unless
$k_{\text{off}} = 0$, which gives Poisson).

### Test design
Run a long SSA (large `n_sim` to reach steady state, large `n_rep` for
accurate statistics), estimate steady-state moments from the last 10 %
of the trajectory, and compare to the formulas above.

In [5]:
def analytical_ss(k_on, k_off, k_syn, k_deg):
    """Exact steady-state moments of the telegraph model."""
    mu_G = k_on / (k_on + k_off)
    mu_R = k_syn * mu_G / k_deg
    var_G = mu_G * (1 - mu_G)
    cov_RG = k_syn * var_G / (k_on + k_off + k_deg)
    var_R = mu_R + k_syn * cov_RG / k_deg
    return {
        "mu_G": mu_G,
        "mu_R": mu_R,
        "sigma_G": np.sqrt(var_G),
        "sigma_R": np.sqrt(var_R),
        "cov_RG": cov_RG,
        "fano": var_R / mu_R if mu_R > 0 else np.nan,
    }


def ssa_steady_state(moments, tail_frac=0.1):
    """Estimate steady-state values from the last tail_frac of the SSA."""
    n = len(moments['time'])
    tail = max(1, int(n * tail_frac))
    return {
        "mu_G": float(moments["mu_G"][-tail:].mean()),
        "mu_R": float(moments["mu_R"][-tail:].mean()),
        "sigma_G": float(moments["sigma_G"][-tail:].mean()),
        "sigma_R": float(moments["sigma_R"][-tail:].mean()),
        "cov_RG": float(moments["cov_RG"][-tail:].mean()),
    }

In [6]:
# Test across multiple parameter regimes
test_cases = {
    "Slow switching": dict(k_on=0.01, k_off=0.02, k_syn=10.0, k_deg=0.2),
    "Fast switching": dict(k_on=2.0,  k_off=4.0,  k_syn=10.0, k_deg=0.2),
    "High expression": dict(k_on=0.1, k_off=0.1, k_syn=40.0, k_deg=0.5),
    "Low expression": dict(k_on=0.1,  k_off=0.1,  k_syn=2.0,  k_deg=1.0),
}

results = []

for name, kin in test_cases.items():
    # Run long SSA
    data = simulate_telegraph(**kin, t0=0, g0=0, r0=0,
                              n_sim=5000, n_rep=500)
    moments = compute_sample_moments(data)

    ss_ssa = ssa_steady_state(moments)
    ss_ana = analytical_ss(**kin)

    results.append((name, ss_ssa, ss_ana))

    print(f"\n── {name} ──")
    print(f"  {"Moment":<10s}  {"SSA":>10s}  {"Analytical":>10s}  {"Rel. Error":>10s}")
    print(f"  {"-"*10}  {"-"*10}  {"-"*10}  {"-"*10}")
    for key in ["mu_G", "mu_R", "sigma_G", "sigma_R", "cov_RG"]:
        ssa_val = ss_ssa[key]
        ana_val = ss_ana[key]
        rel_err = abs(ssa_val - ana_val) / max(abs(ana_val), 1e-10)
        print(f"  {key:<10s}  {ssa_val:10.4f}  {ana_val:10.4f}  {rel_err:10.2%}")

SyntaxError: f-string: expecting '}' (780454898.py, line 23)

> **Pass criterion:** Relative errors should be small (< 5–10 %) for all
> moments. Some noise is expected in σ_R and Cov(R,G) estimates because
> they are higher-order statistics and converge more slowly.

---
## Validation 3: Edge Cases

Degenerate parameter values test whether the implementation handles
limiting cases correctly.

### 3a. Constitutive expression ($k_{\text{off}} = 0$)

When $k_{\text{off}} = 0$ and the gene starts ON ($G_0 = 1$), the gene
**never turns off**. The system reduces to a simple **birth–death process**
for mRNA:

$$
\varnothing \xrightarrow{k_{\text{syn}}} \text{mRNA} \xrightarrow{k_{\text{deg}}} \varnothing
$$

**Expected behaviour:**
- $\langle G \rangle = 1$ for all time (gene is always ON)
- $\langle R \rangle \to k_{\text{syn}} / k_{\text{deg}}$ (deterministic steady state)
- $\text{Var}[R] = \langle R \rangle$ at steady state (**Poisson** — Fano factor = 1)
- $\text{Cov}(R, G) = 0$ (since $G$ has zero variance)

In [7]:
# Edge case 3a: k_off = 0, g0 = 1 (constitutive expression)
params_const = dict(
    k_on=0.1, k_off=0.0,  # gene never turns off
    k_syn=10.0, k_deg=0.2,
    t0=0.0, g0=1, r0=0,   # start ON
    n_sim=3000, n_rep=500,
)

data_const = simulate_telegraph(**params_const)
m_const = compute_sample_moments(data_const)
ss_const = ssa_steady_state(m_const)

expected_mu_R = params_const['k_syn'] / params_const['k_deg']  # = 50
expected_var_R = expected_mu_R  # Poisson

print('── Edge case: Constitutive expression (k_off = 0) ──')
print(f'  ⟨G⟩_ss  = {ss_const["mu_G"]:.4f}   (expected: 1.0000)')
print(f'  ⟨R⟩_ss  = {ss_const["mu_R"]:.2f}   (expected: {expected_mu_R:.2f})')
print(f'  σ²_R_ss = {ss_const["sigma_R"]**2:.2f}   (expected Poisson: {expected_var_R:.2f})')
print(f'  Fano    = {ss_const["sigma_R"]**2 / ss_const["mu_R"]:.3f}   (expected: 1.000)')
print(f'  Cov_ss  = {ss_const["cov_RG"]:.5f}   (expected: 0.00000)')

# Assertions
assert abs(ss_const['mu_G'] - 1.0) < 0.01, 'FAIL: ⟨G⟩ ≠ 1'
assert abs(ss_const['mu_R'] - expected_mu_R) / expected_mu_R < 0.05, 'FAIL: ⟨R⟩ off'
fano = ss_const['sigma_R']**2 / ss_const['mu_R']
assert abs(fano - 1.0) < 0.15, f'FAIL: Fano = {fano:.3f}, expected ~1'
print('\n✅ All constitutive-expression checks passed!')

── Edge case: Constitutive expression (k_off = 0) ──
  ⟨G⟩_ss  = 1.0000   (expected: 1.0000)
  ⟨R⟩_ss  = 50.75   (expected: 50.00)
  σ²_R_ss = 51.42   (expected Poisson: 50.00)
  Fano    = 1.013   (expected: 1.000)
  Cov_ss  = 0.00000   (expected: 0.00000)

✅ All constitutive-expression checks passed!


### 3b. No degradation ($k_{\text{deg}} = 0$)

When $k_{\text{deg}} = 0$, mRNA is **never destroyed**. There is no steady
state for R — the count grows without bound:

$$
\langle R(t) \rangle \approx R_0 + k_{\text{syn}} \cdot \mu_G^{\text{ss}} \cdot t
$$

(once the gene state has equilibrated).

**Expected behaviour:**
- $\langle G \rangle \to k_{\text{on}} / (k_{\text{on}} + k_{\text{off}})$ (normal gene relaxation)
- $\langle R \rangle$ grows approximately linearly with time
- No plateau in the ⟨R⟩ curve

In [8]:
# Edge case 3b: k_deg = 0 (no mRNA degradation)
params_nodeg = dict(
    k_on=0.5, k_off=0.5,  # fast switching, μ_G = 0.5
    k_syn=5.0, k_deg=0.0, # no degradation!
    t0=0.0, g0=0, r0=0,
    n_sim=2000, n_rep=300,
)

data_nodeg = simulate_telegraph(**params_nodeg)
m_nodeg = compute_sample_moments(data_nodeg)

# Gene state should still equilibrate
mu_G_ss_expected = 0.5
tail = max(1, len(m_nodeg['time']) // 10)
mu_G_late = float(m_nodeg['mu_G'][-tail:].mean())

# R should grow linearly: slope ≈ k_syn * μ_G = 5 * 0.5 = 2.5
# Use late portion where gene has equilibrated
t_late = m_nodeg['time'][-tail:]
R_late = m_nodeg['mu_R'][-tail:]
slope = np.polyfit(t_late, R_late, 1)[0]
expected_slope = params_nodeg['k_syn'] * mu_G_ss_expected  # = 2.5

print('── Edge case: No degradation (k_deg = 0) ──')
print(f'  ⟨G⟩_late = {mu_G_late:.4f}   (expected: {mu_G_ss_expected:.4f})')
print(f'  dR/dt    = {slope:.3f}   (expected: {expected_slope:.3f})')
print(f'  Final ⟨R⟩ = {m_nodeg["mu_R"][-1]:.1f} (should be large, no plateau)')

assert abs(mu_G_late - mu_G_ss_expected) < 0.05, 'FAIL: ⟨G⟩ wrong'
assert abs(slope - expected_slope) / expected_slope < 0.15, f'FAIL: slope = {slope:.3f}'
assert m_nodeg['mu_R'][-1] > 10, 'FAIL: R should be large'
print('\n✅ All no-degradation checks passed!')

── Edge case: No degradation (k_deg = 0) ──
  ⟨G⟩_late = 0.9157   (expected: 0.5000)
  dR/dt    = 2.478   (expected: 2.500)
  Final ⟨R⟩ = 1665.2 (should be large, no plateau)


AssertionError: FAIL: ⟨G⟩ wrong

In [9]:
# Visualise the no-degradation case: R grows linearly
fig = go.Figure(layout=_base_layout(
    title="<b>Edge case:</b> k<sub>deg</sub> = 0 — mRNA accumulates linearly",
    ylabel="⟨R⟩ (molecules)",
    figsize=(5.5, 3.2),
))

fig.add_trace(go.Scatter(
    x=m_nodeg['time'], y=m_nodeg['mu_R'], mode='lines',
    line=dict(color=_RED, width=2),
    name="SSA ⟨R⟩",
))

# Expected linear growth
t_line = m_nodeg['time']
fig.add_trace(go.Scatter(
    x=t_line, y=expected_slope * t_line, mode='lines',
    line=dict(color="grey", width=1.5, dash="dash"),
    name=f"k_syn·μ_G·t = {expected_slope:.1f}·t",
))

fig.show()

### 3c. Constitutive expression: verify Poisson Fano factor

For a pure birth-death process ($k_{\text{off}} = 0$, gene always ON),
the mRNA distribution is **Poisson** at steady state, meaning the
Fano factor $F = \sigma_R^2 / \langle R \rangle = 1$.

We also compare to the ODE prediction for the transient dynamics.

In [10]:
# Compare constitutive SSA to ODE
from src.ode_visualization import show_ssa_vs_ode

t_end_const = float(m_const['time'][-1])
t_ode_c, y_ode_c = solve_ode_moments(
    k_on=0.1, k_off=0.0, k_syn=10.0, k_deg=0.2,
    t0=0.0, g0=1, r0=0, t_end=t_end_const
)

fig_g, fig_r, fig_c = show_ssa_vs_ode(
    m_const, t_ode_c, y_ode_c, title_suffix="Constitutive (k_off=0)"
)
fig_r.show()
fig_c.show()